# snATAC-Express Tutorial
### This notebook demonstrates how to use snATAC-Express to predict gene expression from chromatin accessibility data using machine learning.

## Overview
### snATAC-Express runs in two phases:

1. Phase 1: Initial modeling for iterative refinement and feature importance ranking
2. Phase 2: Refined modeling using only the most important peaks (top 95%)

## 1. Setup and Imports

In [13]:
# snATAC-Express Tutorial: End-to-End Example Using run_multi_test

import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Add the package to the path
sys.path.append(os.path.abspath('snatac_express'))

# Import the specific functions we need
from snatac_express.scripts.data_preprocessing import (
    load_peak_input, 
    load_gex_input, 
    get_pseudobulk
)

print("✅ Environment ready!")

✅ Environment ready!


## 2. Configuration and Data Paths

In [14]:
# Define configuration with correct project directory
import os

# Set the correct project directory
project_dir = '/home/maggiebrown/projects/snATAC-Express'
print(f"🏠 Project directory: {project_dir}")

config = {
    'input_dir': os.path.join(project_dir, 'example_data', 'input_data'),
    'output_dir': os.path.join(project_dir, 'results', 'tutorial_bach2'),
    'config_yaml': os.path.join(project_dir, 'config.yaml'),
    'gene': 'BACH2'
}

# Create output directory
os.makedirs(config['output_dir'], exist_ok=True)
print(f"📁 Output directory: {config['output_dir']}")

# Check input data
if os.path.exists(config['input_dir']):
    input_files = os.listdir(config['input_dir'])
    print(f"📂 Input files available:")
    for file in input_files:
        print(f"  - {file}")
else:
    print(f"❌ Input directory not found: {config['input_dir']}")

🏠 Project directory: /home/maggiebrown/projects/snATAC-Express
📁 Output directory: /home/maggiebrown/projects/snATAC-Express/results/tutorial_bach2
📂 Input files available:
  - sparse_gex_matrix_colnames.txt
  - group_coverages.csv
  - sparse_gex_matrix_rownames.txt
  - sparse_peak_matrix_colnames.txt
  - sparse_peak_matrix_rownames.txt
  - sparse_gex_matrix.txt.mtx
  - genelist_genebody.txt
  - sparse_peak_matrix.txt.mtx


## 3. Inspect Config File

In [15]:
# View the configuration file. This is the file that contains the parameters for the analysis and may be edited by the user.
print("Configuration file contents:")
print("=" * 50)
with open(config['config_yaml'], 'r') as f:
    print(f.read())

Configuration file contents:
# snATAC-Express Configuration

# General settings
project_name: "snATAC_Express_Analysis"
output_dir: "results"
n_jobs: -1  # Number of parallel jobs (-1 = use all cores)
random_seed: 12345

# Input data paths
input_data:
  sparse_gex_matrix: "sparse_gex_matrix.txt.mtx"
  sparse_peak_matrix: "sparse_peak_matrix.txt.mtx"
  group_coverages: "group_coverages.csv"
  gene_list: "genelist_genebody.txt"
  
# Phase 1 settings (Initial modeling and feature ranking)
phase1:
  # Pseudobulk settings
  pseudobulk:
    replicate: "1"  # Which replicate to use (1 or 2)
    min_cells: 10   # Minimum cells per pseudobulk group
    
  # Peak filtering options: Peaks in at least X% of cells to include.
  peak_filters:
    - name: "all_peaks"
      min_sample_presence: 0.0
    - name: "peaks_10pct"
      min_sample_presence: 0.1
    - name: "peaks_50pct" 
      min_sample_presence: 0.5
  
  # WHICH PEAK FILTER TO USE (set this to 0, 1, or 2)
  # 0 = all_peaks (use all peaks r

## 4. Peak at ATAC-seq Peak Data

In [16]:
# Load ATAC-seq peak matrix
print("🔍 Loading ATAC-seq peak data...")

# Load peak data using the package function
peak_data = load_peak_input('sparse_peak_matrix.txt.mtx', input_dir=config['input_dir'])

print(f"�� Peak matrix shape: {peak_data.shape}")
print(f"📊 Peak data info:")
print(f"  - Number of peaks: {peak_data.shape[0]}")
print(f"  - Number of cells: {peak_data.shape[1]}")
print(f"  - Data types: {peak_data.dtypes.unique()}")
print(f"  - Memory usage: {peak_data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

# Show sample of peak data
print("\n📋 Sample peak data (first 3 peaks, first 3 cells):")
print(peak_data.iloc[:3, :min(3, peak_data.shape[1])])

# Check if this is test data
if peak_data.shape[1] <= 2:
    print(f"\n⚠️  This appears to be test data with only {peak_data.shape[1]} cell(s)")
    print("   The tutorial will continue but results may be limited")

🔍 Loading ATAC-seq peak data...
�� Peak matrix shape: (247, 76453)
📊 Peak data info:
  - Number of peaks: 247
  - Number of cells: 76453
  - Data types: [dtype('uint8')]
  - Memory usage: 18.03 MB

📋 Sample peak data (first 3 peaks, first 3 cells):
                        Pool_8#GTTAACGGTGCTTTAC-1  Pool_8#CCTTATGTCGCAACAT-1  \
chr6:89827394-89827894                          0                          0   
chr6:89827897-89828397                          0                          0   
chr6:89828900-89829400                          0                          0   

                        Pool_8#GCATATATCAAACTCA-1  
chr6:89827394-89827894                          0  
chr6:89827897-89828397                          0  
chr6:89828900-89829400                          0  


## 5. Peak at Gene Expression Data

In [17]:
print("🧬 Loading gene expression data...")

# Only pass the matrix file name and input_dir
gex_data = load_gex_input('sparse_gex_matrix.txt.mtx', input_dir=config['input_dir'])

print(f"📈 Gene expression matrix shape: {gex_data.shape}")
print(f"📊 Gene expression data info:")
print(f"  - Number of genes: {gex_data.shape[0]}")
print(f"  - Number of cells: {gex_data.shape[1]}")
print(f"  - Data types: {gex_data.dtypes.unique()}")

# Check if BACH2 is in the data
bach2_expression = gex_data.loc[gex_data.index == 'BACH2']
if not bach2_expression.empty:
    print(f"\n🎯 BACH2 expression found!")
    print(f"  - Expression values: {bach2_expression.values.flatten()}")
    print(f"  - Mean expression: {bach2_expression.values.mean():.4f}")
    print(f"  - Std expression: {bach2_expression.values.std():.4f}")
    print(f"  - Number cells with BACH2 transcripts: {(bach2_expression != 0).sum().sum()}")


    # Show sample of GEX data
    print("\n📋 Sample GEX data (BACH2, first 3 cells):")
    print(bach2_expression.iloc[:3, :min(20, bach2_expression.shape[1])])

else:
    print(f"\n⚠️  BACH2 not found in gene expression data")
    print(f"Available genes: {list(gex_data.index)}")

🧬 Loading gene expression data...
📈 Gene expression matrix shape: (1, 76453)
📊 Gene expression data info:
  - Number of genes: 1
  - Number of cells: 76453
  - Data types: [dtype('uint8')]

🎯 BACH2 expression found!
  - Expression values: [ 0  0  0 ...  0 16  0]
  - Mean expression: 10.6663
  - Std expression: 21.1203
  - Number cells with BACH2 transcripts: 34057

📋 Sample GEX data (BACH2, first 3 cells):
       Pool_8#GTTAACGGTGCTTTAC-1  Pool_8#CCTTATGTCGCAACAT-1  \
BACH2                          0                          0   

       Pool_8#GCATATATCAAACTCA-1  Pool_8#CCGTGCTGTAGTTGGC-1  \
BACH2                          0                          0   

       Pool_8#CATAACGGTTATGTGG-1  Pool_8#CTGACCAAGTAAGTCC-1  \
BACH2                          0                          0   

       Pool_8#GGATGGCCAAACCTAT-1  Pool_8#GGAGCAAGTCCTTCTC-1  \
BACH2                          0                          0   

       Pool_8#CTCTGTTCAATTAAGG-1  Pool_8#TTAGGCCCATCATGGC-1  \
BACH2              

## 6. Run the Pipeline

In [18]:
# Import the main workflow runner
import os
import glob
import random

# Set a global random seed for reproducibility
SEED = 12345
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("🚀 Starting snATAC-Express Two-Phase Pipeline...")
print("=" * 50)

# Change to the project directory to ensure relative paths work
original_cwd = os.getcwd()
project_dir = '/home/maggiebrown/projects/snATAC-Express'
os.chdir(project_dir)

try:
    # Set up command line arguments for BOTH phases
    sys.argv = [
        'run_multi_test.py',
        '--config', 'config.yaml',
        '--phase', 'both',  # Run both Phase 1 and Phase 2
        '--gene', 'BACH2'   # Optional: specific gene
    ]
    
    # Import and run the main function
    from snatac_express.scripts.run_multi_test import main as run_workflow
    
    run_workflow()
    print("✅ Two-phase pipeline completed successfully!")
    
except Exception as e:
    print(f"❌ Error during pipeline execution: {e}")
    raise
finally:
    # Change back to original directory
    os.chdir(original_cwd)

2025-06-16 21:32:49,832 - INFO - Starting snATAC-Express workflow
2025-06-16 21:32:49,833 - INFO - Configuration: config.yaml
2025-06-16 21:32:49,833 - INFO - Phase(s) to run: both
2025-06-16 21:32:49,834 - INFO - 
2025-06-16 21:32:49,834 - INFO - PHASE 1: Initial modeling with feature selection
2025-06-16 21:32:49,835 - INFO - ============================================================
2025-06-16 21:32:49,837 - INFO - Loading ATAC peaks...


🚀 Starting snATAC-Express Two-Phase Pipeline...


2025-06-16 21:32:49,984 - INFO - Loading gene expression...
2025-06-16 21:32:50,107 - INFO - Processing 1 genes...
2025-06-16 21:32:50,108 - INFO - Processing gene BACH2
2025-06-16 21:32:50,953 - INFO -   Total peaks: 247
2025-06-16 21:32:50,955 - INFO -   Filtered peaks (≥10% samples): 132
2025-06-16 21:32:50,956 - INFO -   Running random_forest


Average Score (all peaks): 0.5055706908233909


2025-06-16 21:33:01,600 - INFO -     rf_ranker:
2025-06-16 21:33:01,601 - INFO -       All peaks: R² = 0.5056 (132 peaks)
2025-06-16 21:33:01,602 - INFO -       95% peaks: R² = 0.5204 (97 peaks)


...

Average Score (95% peaks): 0.533138828630215
Average Score (all peaks): 0.6302987575531006


2025-06-16 21:45:59,561 - INFO -     xgb_ranker: R² = 0.6303 (114 peaks)
2025-06-16 21:45:59,562 - INFO -   Running lightgbm


Average Score (95% peaks): 0.6143041729927063
Average Score (all peaks): 0.5831298552699568


2025-06-16 21:46:08,435 - INFO -     lgbm_ranker: R² = 0.5831 (114 peaks)
2025-06-16 21:46:08,436 - INFO - Step 5: Creating Phase 2 master aggregated peak ranks
2025-06-16 21:46:08,436 - INFO - Creating Phase 2 master aggregated peak ranks file...
2025-06-16 21:46:08,438 - INFO -   BACH2: 114 peaks for Phase 2
2025-06-16 21:46:08,440 - INFO -   Saved Phase 2 master aggregated peak ranks to results/aggregated_results/master_aggregated_peak_ranks.csv
2025-06-16 21:46:08,442 - INFO -   Saved Phase 2 aggregation summary to results/aggregated_results/phase2_aggregation_summary.txt
2025-06-16 21:46:08,443 - INFO - Step 6: Summarizing Phase 2 results
2025-06-16 21:46:08,445 - INFO - Saved Phase 2 summary to results/phase2_cv_summary.txt
2025-06-16 21:46:08,445 - INFO - 
Phase 2 Summary Statistics:
2025-06-16 21:46:08,446 - INFO - Total genes analyzed: 1
2025-06-16 21:46:08,446 - INFO - Average R²: 0.5767
2025-06-16 21:46:08,447 - INFO - Median R²: 0.5831
2025-06-16 21:46:08,449 - INFO - 
Phas

Average Score (95% peaks): 0.5902080471987255
✅ Two-phase pipeline completed successfully!


## 7. List and explore output files

In [23]:
import os
import glob

# List all files in the results directory
os.chdir('/home/maggiebrown/projects/snATAC-Express')
print("=== Results Directory Structure ===")
for root, dirs, files in os.walk("results"):
    level = root.replace("results", '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for f in files:
        print(f"{subindent}{f}")

=== Results Directory Structure ===
results/
  snATAC_Express_20250616_213249.log
  phase2_cv_summary.txt
  cv_summary.txt
  logs/
  tutorial_bach2/
  aggregated_results/
    master_aggregated_peak_ranks.csv
    selected_peaks_summary.csv
    phase2_aggregation_summary.txt
    phase2_aggregated_summary.csv
    BACH2/
      aggregated_peak_importances_exclLR.csv
  phase1_results/
    BACH2/
      feature_rankings/
        xgb_permranker/
          trained_model_top95_peaks.pkl
          BACH2_28peaks_permranker_importance.csv
          BACH2_132peaks_permranker_importance.csv
          trained_model_all_peaks.pkl
          cross_validations_all_peaks/
            Column_1_Fold_4.csv
            Column_0_Fold_4.csv
            Column_0_Fold_2.csv
            Column_0_Fold_3.csv
            Column_1_Fold_2.csv
            Column_2_Fold_0.csv
            Column_0_Fold_1.csv
            Column_2_Fold_2.csv
            Column_2_Fold_1.csv
            Column_1_Fold_1.csv
            Column_2_

## 8. Display final run summary


In [24]:
import pandas as pd

phase2_summary_path = "results/phase2_cv_summary.txt"
if os.path.exists(phase2_summary_path):
    phase2_summary = pd.read_csv(phase2_summary_path, sep='\t')
    print("=== Phase 2 Cross-Validation Summary ===")
    display(phase2_summary.head(10))  # Show first 10 rows
    print(f"\nTotal genes analyzed: {phase2_summary['Gene'].nunique()}")
    print(f"Methods: {phase2_summary['Method'].unique()}")
else:
    print("Phase 2 summary not found!")

=== Phase 2 Cross-Validation Summary ===


,Gene,Method,nPeaks,Phase,CV_R2
0,BACH2,random_forest_rf_ranker,114,Phase2_Aggregated,0.516760
1,BACH2,xgboost_xgb_ranker,114,Phase2_Aggregated,0.630299
2,BACH2,lightgbm_lgbm_ranker,114,Phase2_Aggregated,0.583130



Total genes analyzed: 1
Methods: ['random_forest_rf_ranker' 'xgboost_xgb_ranker' 'lightgbm_lgbm_ranker']


## 9. Visualize Top Aggregated Peak Importances

In [25]:
# Show top aggregated peaks for a gene (e.g., BACH2)
agg_peaks_path = "results/aggregated_results/BACH2/aggregated_peak_importances_exclLR.csv"
if os.path.exists(agg_peaks_path):
    agg_peaks = pd.read_csv(agg_peaks_path)
    print("=== Top Aggregated Peak Importances (Phase 2, BACH2) ===")
    display(agg_peaks.head(10))
else:
    print("Aggregated peak importances file not found!")

=== Top Aggregated Peak Importances (Phase 2, BACH2) ===


,Unnamed: 0,Peaks,rf_dropcolranker_Zscore,rf_permranker_Zscore,rf_ranker_Zscore,xgb_dropcolranker_Zscore,xgb_permranker_Zscore,xgb_ranker_Zscore,lgbm_dropcolranker_Zscore,lgbm_permranker_Zscore,lgbm_ranker_Zscore,Average_Zscore
0,0,chr6:90304295-90304795,1.890798,7.852147,4.850932,6.729509,9.309196,9.991696,5.307010,9.088362,6.197520,6.801908
1,1,chr6:90315537-90316037,2.366799,7.510255,8.376789,4.442982,6.291134,4.676993,5.463394,6.153286,4.062489,5.482680
2,2,chr6:90080753-90081253,0.379610,0.882994,0.810770,-1.479272,1.172808,2.091992,4.371573,2.433317,1.571620,1.359490
3,3,chr6:90274311-90274811,0.150420,0.665846,0.348106,2.004203,0.405514,0.244312,2.211284,0.368768,2.461216,0.984408
4,4,chr6:89829417-89829917,-0.353047,0.325493,0.184506,3.619057,0.317335,0.217724,-0.484402,0.241893,3.172893,0.804606
5,5,chr6:89952722-89953222,0.072472,0.139345,-0.194870,0.199113,-0.073403,-0.073817,4.081881,0.429004,2.461216,0.782327
6,6,chr6:90295074-90295574,0.986365,-0.163238,0.284384,0.444896,0.012935,-0.012670,2.192056,0.664914,1.927458,0.704122
7,7,chr6:90383191-90383691,-0.431003,-0.011885,0.421852,1.584745,0.041594,-0.029012,0.735925,0.087820,2.283297,0.520370
8,8,chr6:90216189-90216689,0.291303,0.909386,1.848964,-0.640385,-0.121461,-0.167839,1.112608,-0.026575,0.682024,0.432003
9,9,chr6:90375624-90376124,-1.088303,1.275699,1.886153,-0.668207,-0.129459,-0.134069,0.376874,0.087177,1.927458,0.392591


# 10. Final model summary

In [26]:
agg_summary_path = "results/aggregated_results/phase2_aggregated_summary.csv"
if os.path.exists(agg_summary_path):
    agg_summary = pd.read_csv(agg_summary_path)
    print("=== Phase 2 Aggregated Summary ===")
    display(agg_summary.head(10))
else:
    print("Phase 2 aggregated summary not found!")

=== Phase 2 Aggregated Summary ===


,gene,n_peaks_phase2,avg_r2_phase2,best_method,best_r2_phase2,n_methods
0,BACH2,114,0.57673,XGB_xgb_ranker,0.630299,3
